# Employee Attrition Prediction System
**Author:** Data Science Team  
**Objective:** Build a predictive machine learning classification model to determine whether an employee is likely to leave an organization (attrition).

### Project Methodology (Machine Learning Lifecycle):
1. **Problem Understanding**: Identify business goals and study characteristics of the dataset.
2. **Data Collection**: Fetch the IBM HR Analytics dataset (1470 records, 35 attributes).
3. **Data Preprocessing**: Handle duplicates, analyze missing values, treat outliers, encode features, scale numbers.
4. **Exploratory Data Analysis (EDA)**: Conduct univariate, bivariate, and multivariate analysis with visual representations.
5. **Feature Engineering**: Select important features and address class imbalance using SMOTE.
6. **Model Building**: Train Logistic Regression, Linear Regression, Random Forest, and XGBoost models.
7. **Model Evaluation**: Compare models using Accuracy, Precision, Recall, F1-Score, and ROC-AUC metrics.
8. **Deployment**: Package models and build a Streamlit web application.


## Setup & Package Ingestion
First, we import the core data science, plotting, and machine learning modules.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline


## Data Collection
Let's load the dataset and view the basic structure of the data.


In [ ]:
# Load the downloaded dataset
dataset_path = '../Dataset/WA_Fn-UseC_-HR-Employee-Attrition.csv'
if not os.path.exists(dataset_path):
    dataset_path = 'Dataset/WA_Fn-UseC_-HR-Employee-Attrition.csv' # Fallback for relative paths

df = pd.read_csv(dataset_path)
print(f'Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()


In [ ]:
# Check basic info and data types
df.info()


## Data Preprocessing
We clean the data by checking for duplicates, missing values, dropping non-informative columns, and capping outliers.


In [ ]:
# Check for duplicates and missing values
print('Duplicates:', df.duplicated().sum())
print('Missing Values:\n', df.isnull().sum().sum())


In [ ]:
# Analyze feature statistics
df.describe().T


In [ ]:
# Identify constant columns
for col in df.columns:
    if df[col].nunique() == 1:
        print(f'Constant Column: {col} (Value: {df[col].iloc[0]})')
        
# Drop EmployeeNumber as it is a unique identifier and not predictive
print('Dropping useless columns: EmployeeCount, StandardHours, Over18, EmployeeNumber')
df.drop(columns=['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber'], errors='ignore', inplace=True)


In [ ]:
# Treat Outliers (Capping continuous variables using IQR)
outlier_cols = ['MonthlyIncome', 'TotalWorkingYears', 'YearsAtCompany', 
                'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']

def treat_outliers(data, columns):
    cleaned_data = data.copy()
    for col in columns:
        q1 = cleaned_data[col].quantile(0.25)
        q3 = cleaned_data[col].quantile(0.75)
        iqr = q3 - q1
        lower_limit = q1 - 1.5 * iqr
        upper_limit = q3 + 1.5 * iqr
        cleaned_data[col] = cleaned_data[col].clip(lower=lower_limit, upper=upper_limit)
    return cleaned_data

print('Original statistics for MonthlyIncome outliers:')
print(df['MonthlyIncome'].describe())
df = treat_outliers(df, outlier_cols)
print('\nCapped statistics for MonthlyIncome outliers:')
print(df['MonthlyIncome'].describe())


## Exploratory Data Analysis (EDA)
We visualize features individually (univariate), in relationships (bivariate), and analyze correlations.


### Univariate Analysis: Target Distribution


In [ ]:
# Target variable class distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='Attrition', data=df)
plt.title('Employee Attrition Distribution (Imbalanced Class)')
plt.xlabel('Attrition status')
plt.ylabel('Count')
plt.show()
print('Attrition percentages:')
print(df['Attrition'].value_counts(normalize=True) * 100)


### Univariate Analysis: Age Distribution


In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df['Age'], kde=True, bins=30)
plt.title('Distribution of Employee Age')
plt.xlabel('Age')
plt.show()


### Bivariate Analysis: Income by Attrition


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='Attrition', y='MonthlyIncome', data=df)
plt.title('Monthly Income vs. Attrition Status')
plt.xlabel('Attrition')
plt.ylabel('Monthly Income ($)')
plt.show()


### Bivariate Analysis: Overtime vs Attrition


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='OverTime', hue='Attrition', data=df)
plt.title('OverTime impact on Attrition')
plt.xlabel('Works OverTime')
plt.ylabel('Count')
plt.legend(title='Attrition')
plt.show()


### Correlation Analysis


In [ ]:
# Correlation heatmap for numerical features
numerical_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(12, 10))
sns.heatmap(numerical_df.corr(), annot=False, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap of Employee Metrics')
plt.show()


## Feature Engineering and Data Balancing
We divide data into features and target, set up hot-encoding for categorical variables, scaling for numerical variables, and apply SMOTE to balance the dataset.


In [ ]:
# Target & Features split
y = df['Attrition'].apply(lambda x: 1 if x == 'Yes' else 0)
X = df.drop(columns=['Attrition'])

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

print('Numerical Features:', len(num_cols))
print('Categorical Features:', len(cat_cols))


In [ ]:
# Train/Test split (80/20, stratify to maintain target class distribution)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')


In [ ]:
# Setup Preprocessor pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

cat_encoder = preprocessor.named_transformers_['cat']
cat_feature_names = cat_encoder.get_feature_names_out(cat_cols).tolist()
feature_names = num_cols + cat_feature_names
print(f'Total Features post-encoding: {len(feature_names)}')


In [ ]:
# Data Balancing: Apply SMOTE on processed training data only
print('Pre-SMOTE Train Target counts:')
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_processed, y_train)

print('\nPost-SMOTE Train Target counts:')
print(pd.Series(y_train_res).value_counts())


## Model Building
We build and train four models as classifier baselines:
1. **Logistic Regression**
2. **Linear Regression (thresholded at 0.5)**
3. **Random Forest Classifier**
4. **XGBoost Classifier**


In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.08, use_label_encoder=False, eval_metric='logloss', random_state=42)
}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_res, y_train_res)


## Model Evaluation
We evaluate models on the test dataset using metrics like Accuracy, Precision, Recall, F1-Score, and ROC-AUC. We also examine Confusion Matrices and ROC curves.


In [ ]:
metrics = {}
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
roc_fig, roc_ax = plt.subplots(figsize=(8, 6))

for idx, (name, model) in enumerate(models.items()):
    # Handle classification vs regression output
    if hasattr(model, 'predict_proba'):
        y_pred = model.predict(X_test_processed)
        y_prob = model.predict_proba(X_test_processed)[:, 1]
    else:
        y_prob = model.predict(X_test_processed)
        y_prob = np.clip(y_prob, 0, 1)
        y_pred = (y_prob >= 0.5).astype(int)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    
    metrics[name] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': roc_auc
    }
    
    # Plot Confusion Matrix in subplot
    cm = confusion_matrix(y_test, y_pred)
    ax = axes.flatten()[idx]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Stayed', 'Left'], yticklabels=['Stayed', 'Left'])
    ax.set_title(f'{name} Confusion Matrix')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
    
    # Plot ROC
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_ax.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})')

fig.tight_layout()
plt.show()

# Finalize ROC plot
roc_ax.plot([0, 1], [0, 1], 'k--', label='Random')
roc_ax.set_xlabel('False Positive Rate')
roc_ax.set_ylabel('True Positive Rate')
roc_ax.set_title('ROC Curve Comparison')
roc_ax.legend(loc='lower right')
plt.show()


In [ ]:
# Performance Metrics DataFrame
df_metrics = pd.DataFrame(metrics).T
df_metrics


### Top Predictors of Attrition


In [ ]:
# Plot Top 10 Features from Random Forest
rf_model = models['Random Forest']
rf_importances = rf_model.feature_importances_
indices = np.argsort(rf_importances)[::-1][:10]

plt.figure(figsize=(10, 5))
sns.barplot(x=rf_importances[indices], y=np.array(feature_names)[indices], palette='viridis')
plt.title('Top 10 Feature Importances (Random Forest)')
plt.xlabel('Importance')
plt.show()
